<a href="https://colab.research.google.com/github/damlattlcn/Crowd-Counting-CSRNet-VGG16/blob/main/kalabal%C4%B1k_tespiti_e%C4%9Fitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Drive bağlantısını yapma:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/crowd_detection

/content/drive/MyDrive/crowd_detection


In [ ]:
# Gerekli kütüphaneleri yükleme:
!pip install tensorflow
!pip install opencv-python
!pip install numpy
!pip install matplotlib
!pip install keras

In [ ]:
!pip install torch torchvision
!pip install matplotlib
!pip install opencv-python-headless numpy matplotlib

In [ ]:
import os
import cv2
import pandas as pd
import sys
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from urllib.parse import unquote, urlparse
from urllib.error import HTTPError
from zipfile import ZipFile
import tarfile
import shutil
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error


import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import transforms
from torchvision import models

import time
import concurrent.futures
from tqdm import tqdm
import pandas as pd
import PIL
from PIL import Image
from PIL.ImageDraw import Draw
from scipy.ndimage import gaussian_filter

In [ ]:
# Veri setinin yolunu tanımlama:
data_dir = '/content/drive/MyDrive/crowd_detection/'
frames_dir = os.path.join(data_dir, 'frames')
labels_path = os.path.join(data_dir, 'yeni_labels.csv')

# Etiketleri yükleme:
labels = pd.read_csv(labels_path)
print(labels.head())

      Image Names  Yeni Sütun
0  seq_000001.jpg          35
1  seq_000002.jpg          41
2  seq_000003.jpg          41
3  seq_000004.jpg          44
4  seq_000005.jpg          41


In [ ]:
yeni_sutun_isimleri = ['ID', 'Count']

labels.columns = yeni_sutun_isimleri

In [ ]:
labels.head()

,ID,Count
0,seq_000001.jpg,35
1,seq_000002.jpg,41
2,seq_000003.jpg,41
3,seq_000004.jpg,44
4,seq_000005.jpg,41


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

import tensorflow as tf
from tensorflow.keras import layers, models

def create_complex_csrnet(input_shape=(256, 256, 3)):
    # VGG16 frontend
    vgg16 = tf.keras.applications.VGG16(include_top=False, weights='imagenet', input_shape=input_shape)
    frontend = models.Model(inputs=vgg16.input, outputs=vgg16.get_layer('block5_conv3').output)

    # Backend with dilated convolutions and additional complexity
    backend = models.Sequential()

    # İlk katmanı koruyalım
    backend.add(layers.Conv2D(512, (3, 3), dilation_rate=2, activation='relu', padding='same', input_shape=(None, None, 512)))
    backend.add(layers.BatchNormalization())
    backend.add(layers.Dropout(0.3))

    backend.add(layers.Conv2D(512, (3, 3), dilation_rate=2, activation='relu', padding='same'))
    backend.add(layers.BatchNormalization())
    backend.add(layers.Dropout(0.3))

    backend.add(layers.Conv2D(512, (3, 3), dilation_rate=2, activation='relu', padding='same'))
    backend.add(layers.BatchNormalization())
    backend.add(layers.Dropout(0.4))

    backend.add(layers.Conv2D(256, (3, 3), dilation_rate=2, activation='relu', padding='same'))
    backend.add(layers.BatchNormalization())
    backend.add(layers.Dropout(0.4))

    backend.add(layers.Conv2D(128, (3, 3), dilation_rate=2, activation='relu', padding='same'))
    backend.add(layers.BatchNormalization())
    backend.add(layers.Dropout(0.4))

    backend.add(layers.Conv2D(64, (3, 3), dilation_rate=2, activation='relu', padding='same'))
    backend.add(layers.BatchNormalization())
    backend.add(layers.Dropout(0.5))

    # Çıkış katmanı: 32x32 boyutunda yoğunluk haritası
    # Use UpSampling2D to ensure the output is the same size as the target
    backend.add(layers.UpSampling2D(size=(2, 2))) # Upsample to match target size
    backend.add(layers.Conv2D(1, (1, 1), activation='sigmoid', padding='same'))

    # Combine frontend and backend
    x = backend(frontend.output)
    model = models.Model(inputs=frontend.input, outputs=x)

    return model

# Model oluşturma
model = create_complex_csrnet(input_shape=(256, 256, 3))
model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "functional_21"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 256, 256, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_conv1 (Conv2D)                │ (None, 256, 256, 64)        │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_conv2 (Conv2D)                │ (None, 256, 256, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_pool (MaxPooling2D)           │ (None, 128, 128, 64)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv1 (Conv2D)                │ (None, 128, 128, 128)       │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv2 (Conv2D)                │ (None, 128, 128, 128)       │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_pool (MaxPooling2D)           │ (None, 64, 64, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv1 (Conv2D)                │ (None, 64, 64, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv2 (Conv2D)                │ (None, 64, 64, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv3 (Conv2D)                │ (None, 64, 64, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_pool (MaxPooling2D)           │ (None, 32, 32, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv1 (Conv2D)                │ (None, 32, 32, 512)         │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv2 (Conv2D)                │ (None, 32, 32, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv3 (Conv2D)                │ (None, 32, 32, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_pool (MaxPooling2D)           │ (None, 16, 16, 512)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv1 (Conv2D)                │ (None, 16, 16, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv2 (Conv2D)                │ (None, 16, 16, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv3 (Conv2D)                │ (None, 16, 16, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential (Sequential)              │ (None, 32, 32, 1)           │       8,636,161 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 23,350,849 (89.08 MB)

 Trainable params: 23,346,881 (89.06 MB)

 Non-trainable params: 3,968 (15.50 KB)

In [ ]:
def load_data(frames_dir, labels):
    X, y = [], []
    for idx, row in labels.iterrows():
        img_path = os.path.join(frames_dir, row['ID'])
        img = cv2.imread(img_path)
        img = cv2.resize(img, (256, 256))
        img = img.astype(np.float32) / 255.0  # Normalizasyon
        X.append(img)
        y.append(row['Count'])  # İnsan sayısı

    X = np.array(X)
    y = np.array(y)

    # Yoğunluk haritalarını oluşturma (32x32 boyutunda)
    y_density_maps = []
    for count in y:
        density_map = np.zeros((32, 32), dtype=np.float32)  # Yoğunluk haritasını 32x32 boyutuna ayarlıyoruz
        # Eğer insan sayısı varsa, Gaussian dağılımı ekleyelim
        if count > 0:
            # Gaussian dağılımını yerleştirmek için merkezleme
            x, y = np.mgrid[0:32, 0:32]
            center_x, center_y = np.random.randint(0, 32, size=2)  # Rasgele bir merkez
            gaussian = np.exp(-((x - center_x)**2 + (y - center_y)**2) / (2 * (2**2)))  # Sigma = 2
            density_map += gaussian * count / np.sum(gaussian)  # Normalize et

        y_density_maps.append(density_map)

    y_density_maps = np.array(y_density_maps)
    y_density_maps = np.expand_dims(y_density_maps, axis=-1)  # Kanal boyutunu ekliyoruz
    return X, y_density_maps

# Verileri yükleme
X, y_density_maps = load_data(frames_dir, labels)


In [ ]:
# Verileri eğitim ve test setlerine ayırma
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y_density_maps, test_size=0.2, random_state=42)

In [ ]:
# Modeli derleme
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='mse',
              metrics=['mae'])

In [ ]:
# Modeli eğitme
history = model.fit(X_train, y_train, epochs=50, batch_size=8, validation_data=(X_test, y_test))

Epoch 1/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 104s 313ms/step - loss: 0.2912 - mae: 0.4388 - val_loss: 0.0735 - val_mae: 0.1647
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 93s 238ms/step - loss: 0.1250 - mae: 0.2492 - val_loss: 0.0838 - val_mae: 0.1419
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 80s 229ms/step - loss: 0.1017 - mae: 0.2018 - val_loss: 0.0833 - val_mae: 0.1379
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 44s 222ms/step - loss: 0.0931 - mae: 0.1772 - val_loss: 0.0826 - val_mae: 0.1304
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 85s 238ms/step - loss: 0.0885 - mae: 0.1602 - val_loss: 0.0822 - val_mae: 0.1252
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 82s 238ms/step - loss: 0.0862 - mae: 0.1489 - val_loss: 0.0821 - val_mae: 0.1210
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 78s 220ms/step - loss: 0.0840 - mae: 0.1401 - val_loss: 0.0819 - val_mae: 0.1164
Epoch 8/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 224ms/step - loss: 0.0838 - mae: 0.1340 - val_loss: 0.0819 - val_mae: 0.1138
Epoch 9/50
200/200 ━━━━━━━━━━━━

In [ ]:
# MSE ve MAE hesaplama
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test.flatten(), y_pred.flatten())
mae = mean_absolute_error(y_test.flatten(), y_pred.flatten())

print(f"Test MSE: {mse}")
print(f"Test MAE: {mae}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step
Test MSE: 0.08187147229909897
Test MAE: 0.10257814079523087


In [ ]:
# Modeli kaydetme
model_save_path = '/content/drive/MyDrive/crowd_detection/csrnet_model_6.h5'
model.save(model_save_path)
print(f"Model kaydedildi: {model_save_path}")

Model kaydedildi: /content/drive/MyDrive/crowd_detection/csrnet_model_6.h5
